# Session 6 — TPU: Model Depth and Memory Limits

## Background

Sessions 1–5 measured a single Transformer encoder block. Production models stack many: BERT-base has 12 layers, GPT-2 up to 48. Activation memory scales linearly with depth. At some stack depth and batch size, the TPU's 16 GB HBM2 fills up. The question is not whether OOM eventually happens, but where the boundary is — and whether the TPU's more efficient HBM2 memory layout provides meaningfully more usable depth at `batch=64`.

## Goal

Sweep `n_layers` at `batch=64` on the TPU v5litepod-1. Observe whether OOM occurs within the same range as the GPU. Produce `results/tpu_depth.json` for comparison in `03-analysis.ipynb`.

## Question

How many Transformer layers can the TPU v5litepod-1 sustain at `batch=64` before running out of memory, and does the TPU extend the usable depth range beyond the GPU?

---

**Runtime:** TPU (v5litepod-1 or equivalent)

After running, results are saved to `results/tpu_depth.json` and loaded automatically by `03-analysis.ipynb`.

In [ ]:
import torch
import torch.nn as nn
import torch_xla.core.xla_model as xm
import torch_xla
import time

device = xm.xla_device()
print(f"Device    : {device}")
print(f"XLA HW    : {xm.xla_device_hw(device)}")
print(f"PyTorch   : {torch.__version__}")
print(f"torch_xla : {torch_xla.__version__}")

In [ ]:
import sys; sys.path.insert(0, "..")
from transformer_block import BenchmarkTransformerBlock, D_MODEL, N_HEAD, DIM_FEEDFORWARD

# ── Session 6 config ──────────────────────────────────────────────────────────
BATCH_SIZE     = 64
SEQ_LEN        = 128
N_LAYERS_SWEEP = [1, 2, 4, 6, 8, 12, 16, 24]
N_STEPS        = 20
WARMUP         = 5

class DeepTransformerModel(nn.Module):
    """Stack of N BenchmarkTransformerBlock instances."""
    def __init__(self, n_layers: int):
        super().__init__()
        self.layers = nn.ModuleList([
            BenchmarkTransformerBlock(
                d_model=D_MODEL, nhead=N_HEAD, dim_feedforward=DIM_FEEDFORWARD
            )
            for _ in range(n_layers)
        ])

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        for layer in self.layers:
            x = layer(x)
        return x

print(f"Config  : batch={BATCH_SIZE}, seq_len={SEQ_LEN}, d_model={D_MODEL}")
print(f"Sweep   : n_layers = {N_LAYERS_SWEEP}")
print(f"Steps   : {N_STEPS} (+ {WARMUP} warmup)")
print("DeepTransformerModel defined.")

In [ ]:
def benchmark(n_layers):
    """Returns (throughput, latency_ms) or ('OOM', None) on memory error."""
    try:
        model     = DeepTransformerModel(n_layers).to(device)
        optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
        criterion = nn.MSELoss()
        model.train()

        elapsed = 0.0
        for step in range(N_STEPS + WARMUP):
            x = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL, device=device)
            y = torch.randn(BATCH_SIZE, SEQ_LEN, D_MODEL, device=device)

            torch_xla.sync()
            t0 = time.perf_counter()
            optimizer.zero_grad()
            loss = criterion(model(x), y)
            loss.backward()
            optimizer.step()
            torch_xla.sync()
            t1 = time.perf_counter()

            if step >= WARMUP:
                elapsed += t1 - t0

        del model
        throughput = (N_STEPS * BATCH_SIZE) / elapsed
        latency_ms = (elapsed / N_STEPS) * 1000
        return round(throughput, 1), round(latency_ms, 2)

    except Exception as e:
        # XLA OOM surfaces as a RuntimeError from the XLA runtime
        if "out of memory" in str(e).lower() or "resource_exhausted" in str(e).lower():
            return "OOM", None
        raise

print("Benchmark function defined.")

In [ ]:
results = {}

print(f"{'n_layers':<10}  {'throughput':>15}  {'latency':>12}")
print(f"{'─'*10}  {'─'*15}  {'─'*12}")

oom_hit = False
for nl in N_LAYERS_SWEEP:
    if oom_hit:
        results[str(nl)] = "OOM"
        print(f"{nl:<10}  {'OOM (skipped)':>15}  {'—':>12}")
        continue

    tput, lat = benchmark(nl)
    if tput == "OOM":
        results[str(nl)] = "OOM"
        oom_hit = True
        print(f"{nl:<10}  {'OOM':>15}  {'—':>12}")
    else:
        results[str(nl)] = {"throughput": tput, "latency_ms": lat}
        print(f"{nl:<10}  {tput:>14,.0f}  {lat:>11.1f} ms")

print()
oom_layers = [nl for nl in N_LAYERS_SWEEP if results.get(str(nl)) == "OOM"]
if oom_layers:
    print(f"OOM boundary: first OOM at n_layers={oom_layers[0]}")
else:
    print("No OOM observed across the full sweep.")

In [ ]:
import json, os
from datetime import datetime, timezone

os.makedirs("results", exist_ok=True)
payload = {
    "hardware":       "TPU",
    "device_name":    str(xm.xla_device_hw(device)),
    "session":        "6",
    "batch_size":     BATCH_SIZE,
    "seq_len":        SEQ_LEN,
    "n_layers_sweep": N_LAYERS_SWEEP,
    "timestamp":      datetime.now(timezone.utc).isoformat(),
    "results":        results,
}
path = "results/tpu_depth.json"
with open(path, "w") as f:
    json.dump(payload, f, indent=2)
print(f"Saved → {path}")